# Kuroshio R5.7 — 原冻结表示 × 混合架构的跨种子稳定性资格实验

**研究目的**：R5.6 中 DP-B2 未通过；原 Original 上 R53 ChannelConv-PCConcat 相对同规模 LSTM 的收益不稳定。保持 **原始冻结双分支**、**EOF 99% train-only**、**PC-MSE**、**180 天输入与 +30/60/90 天输出**不变。此次不新增模型结构，不再生成 DP。

**运行环境**：VS Code + WSL + RTX 5080；读取本地 `/mnt/d/KuroshioDaily1993_2025/final/...`。

**阶段一**：2008/2016 的 3 个新随机种子 31/32/33，三个对照（Matched-LSTM、R53-CapacityMatchedLSTM、R53-ChannelConv-PCConcat），另含原 EOF-TCN-LSTM 作为论文保留对照；先完整重测可复现性。**只有**通过预注册 Gate 才执行 **阶段二**：2010/2018 的额外确认；结果不能声称全数据盲测。没有候选则正常结束。

**任何结果都不覆盖原冻结文件。** 自动输出 CSV、中文及英文图表、资格 JSON、ZIP、SHA256、Notebook 内文件链接。完成后上传 `R57_上传给ChatGPT_稳定性资格实验结果.zip`。

In [ ]:
# ===== 0. 配置：先核实三个目录；改好后从头顺序运行 =====
from dataclasses import dataclass, asdict
from pathlib import Path
import os, json, math, gc, time, random, hashlib, warnings, sys
import numpy as np
import pandas as pd

@dataclass
class CFG:
    ROOT: str = '/mnt/d/KuroshioDaily1993_2025'
    DATASET_REL: str = 'final/dataset/daily_dual_branch_paths_1993_2025_FINAL.npz'
    HANDOFF_REL: str = 'final/reports/LONG_DAILY_1993_2025_HANDOFF.json'
    QC_INDEX_REL: str = 'final/reports/最终逐日路径索引.csv'
    MANUAL_QUEUE_REL: str = 'reports/07_人工复核队列.csv'
    MANUAL_METADATA_REL: str = 'manual_qc/metadata'
    OUTPUT_REL: str = 'R5_7_原表示混合模型稳定性资格实验'
    HISTORY: int = 180
    LEADS: tuple = (30, 60, 90)
    TEST_YEARS: tuple = (2005, 2009, 2013, 2017, 2021, 2024)
    PURGE_DAYS: int = 7
    MAX_INPUT_GAP: int = 2
    MAX_MISSING_FRAC: float = 0.03
    MIN_DATA_YEARS: int = 25
    MIN_ANNUAL_COVERAGE: float = 0.85
    MIN_TRAIN: int = 500
    MIN_VAL: int = 100
    MIN_TEST: int = 100
    DROP_BRANCH1_FIRST: int = 12  # 43->31; same old representation
    BRANCH2_MAX_LON: float = 145.0
    EOF_VARIANCE_TARGET: float = 0.99
    BATCH_SIZE: int = 32
    EPOCHS_B1: int = 70
    EPOCHS_B2: int = 70
    PATIENCE: int = 12
    LR: float = 0.001
    WEIGHT_DECAY: float = 0.0001
    DROPOUT: float = 0.30
    SEEDS: tuple = (20,21,22,23,24)
    AR_LAGS: tuple = (3, 7)
    AR_ALPHAS: tuple = (0.1, 1., 10.)
    CLIM_HALF_WINDOW_DAYS: int = 15
    CLIM_MIN_SAMPLES: int = 10
    BOOT_N: int = 1000
    BOOT_BLOCK_DAYS: int = 30
    FORCE_TRAIN: bool = False
    USE_AMP: bool = True
    RUN_FULL_TRAINING: bool = False   # R5.3仅机制资格实验，严禁直接转长训
    TRAIN_OTHER_HYBRIDS: bool = False # 下一轮网络结构比较可启用，不改baseline主表
    FIG_DPI: int = 300
    PILOT_TEST_YEARS: tuple = (2009, 2017)  # QA: 2008/2016, inner early-stop: 2007/2015
    PILOT_SEEDS: tuple = (31, 32, 33)
    PILOT_EPOCHS: int = 24
    PILOT_PATIENCE: int = 7
    GATE_MIN_GAIN_KM: float = 0.50
    GATE_MAX_LEAD_REGRESSION_KM: float = 2.0

    CONFIRM_PILOT_YEARS: tuple = (2013, 2021)  # QA 2012/2020; 曾经被研发流程涉及，不能称独立盲测
    MAX_PARAM_OVERHEAD_RATIO: float = 0.15
    REQUIRE_ALL_LEADS_NONINFERIOR: bool = True
    REQUIRE_BOTH_QA_YEARS_GAIN: bool = True
    ARCHIVE_INCLUDE_WEIGHTS: bool = False
    ARCHIVE_INCLUDE_CACHE: bool = False
    ARCHIVE_MAX_FILE_MB: int = 150
    RUN_CONFIRMATION: bool = True   # 仅阶段一通过才运行阶段二；不自动正式长训
    RAW_TRACK_DIR_REL: str = 'final/tracks_final'
    MAX_MISSING_DP_DAY_FRAC: float = 0.005
    MAX_DP_TEMPORAL_P95_RATIO: float = 1.40
    RAW_EVAL_ISSUES_PER_QA_YEAR: int = 36
    REBUILD_DP: bool = False
    DP_REPORT_EVERY: int = 250
    ARCHIVE_INCLUDE_WEIGHTS: bool = False
    ARCHIVE_INCLUDE_CACHE: bool = False

cfg=CFG(); ROOT=Path(cfg.ROOT); DS=ROOT/cfg.DATASET_REL; OUT=ROOT/cfg.OUTPUT_REL
DATA=OUT/'实验数据'; CKPT=OUT/'模型权重'; PRED=OUT/'逐样本预测'; REP=OUT/'报告'; FZH=OUT/'图表'/'中文'; FEN=OUT/'图表'/'英文'
for d in (DATA,CKPT,PRED,REP,FZH,FEN): d.mkdir(parents=True,exist_ok=True)
print(json.dumps(asdict(cfg),indent=2,ensure_ascii=False))
ACTIVE_REP='Original'
RAW_TRACK_DIR=ROOT/cfg.RAW_TRACK_DIR_REL
if not RAW_TRACK_DIR.is_dir(): raise FileNotFoundError(f'找不到原始完整流轴目录: {RAW_TRACK_DIR}')
DERIVED=OUT/'派生表示_不覆盖冻结数据';DERIVED.mkdir(exist_ok=True,parents=True)
ARCHIVE_PATH=ROOT/'R57_上传给ChatGPT_稳定性资格实验结果.zip'
ARCHIVE_SHA=Path(str(ARCHIVE_PATH)+'.sha256')
if not DS.exists(): raise FileNotFoundError(f'未发现真正的冻结数据：{DS}。请先运行人工QC notebook最后的冻结与数据构建单元。')
# R5.7 不启用 DP，不改变冻结目标。训练合同使用资产内容 SHA256。
ACTIVE_REP='Original'
A2_ORIGINAL=None
sign=hashlib.sha256(DS.read_bytes()).hexdigest()

In [ ]:
# ===== 1. GPU与字体/依赖检查（不合格立即停止，避免白训练）=====
import importlib.util
required=('torch','scipy','sklearn','matplotlib','tqdm','pandas','numpy')
missing=[v for v in required if importlib.util.find_spec(v) is None]
if missing: raise RuntimeError('WSL环境缺依赖：'+', '.join(missing)+'；用当前Kernel对应python执行 python -m pip install torch scipy scikit-learn matplotlib tqdm pandas numpy')
import torch, torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from scipy.spatial import cKDTree
from sklearn.linear_model import Ridge
import matplotlib, matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.font_manager import FontProperties
from tqdm.auto import tqdm
print('kernel=',sys.executable,'torch=',torch.__version__,'CUDA=',torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('本轮RTX5080正式训练要求CUDA GPU；请检查WSL nvidia-smi 与 PyTorch CUDA构建。')
print('GPU=',torch.cuda.get_device_name(0)); DEV=torch.device('cuda')
torch.set_float32_matmul_precision('high')
font_paths=['/mnt/c/Windows/Fonts/msyh.ttc','/mnt/c/Windows/Fonts/simhei.ttf','/mnt/c/Windows/Fonts/simsun.ttc','/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc']
CJK_NAME=None
for path in font_paths:
    if Path(path).exists():
        try:
            font_manager.fontManager.addfont(path)
            CJK_NAME=FontProperties(fname=path).get_name(); break
        except Exception: pass
if CJK_NAME is None:
    raise RuntimeError('中文字体缺失：请在WSL安装 sudo apt install fonts-noto-cjk 或检查 Windows /mnt/c/Windows/Fonts，避免绘图乱码。')
print('Chinese font=',CJK_NAME)

In [ ]:
# ===== 2. 最终数据读取、年度覆盖、人工QC冻结核对 =====
with np.load(DS,allow_pickle=False) as z:
    dates=pd.DatetimeIndex(pd.to_datetime(z['dates'])).normalize()
    b1=np.asarray(z['branch1_lon'],np.float32)
    lat1=np.asarray(z['branch1_target_lats'],float)
    b2=np.asarray(z['branch2_lat'],np.float32)
    lon2=np.asarray(z['branch2_target_lons'],float)
    source=np.asarray(z['source']).astype(str) if 'source' in z.files else np.array(['unknown']*len(dates))
if not (len(dates)==len(b1)==len(b2)==len(source)): raise RuntimeError('dates / 两分支 / source 数量不一致')
if dates.has_duplicates: raise RuntimeError('发现重复日期，禁止将其直接用于训练')
ord=np.argsort(dates.values); dates=dates[ord]; b1=b1[ord];b2=b2[ord];source=source[ord]
if not np.isfinite(b1).all() or not np.isfinite(b2).all(): raise RuntimeError('冻结数据中含 NaN/Inf')

# 物理范围异常只作质控警示（不随意删除或修改路径），便于发现断面插值外推造成的不可信日
suspicious=(~((b1>=119)&(b1<=140)).all(axis=1)) | (~((b2>=15)&(b2<=50)).all(axis=1))
if suspicious.any():
    suspect_df=pd.DataFrame({'异常日期':dates[suspicious], '原因':'双分支坐标超出宽松物理范围，需人工核对'})
    suspect_df.to_csv(DATA/'01_待人工复查的物理范围异常路径.csv',index=False,encoding='utf-8-sig')
    warnings.warn(f'发现 {suspicious.sum()} 天路径越过宽松物理边界，建议正式训练前核实此文件')

if b1.shape[1]<=cfg.DROP_BRANCH1_FIRST: raise RuntimeError('分支1维数过小')
b1=b1[:,cfg.DROP_BRANCH1_FIRST:];lat1=lat1[cfg.DROP_BRANCH1_FIRST:]
keep2=lon2<=cfg.BRANCH2_MAX_LON+1e-8;b2=b2[:,keep2];lon2=lon2[keep2]
if len(b1)!=len(b2) or len(lat1)!=b1.shape[1] or len(lon2)!=b2.shape[1]: raise RuntimeError('双分支物理坐标不匹配')
if dates.min().year>1993 or dates.max().year<2024: raise RuntimeError('数据不覆盖1993–2024所需区间，不可宣称多年全历史验证')
cal=pd.date_range(dates.min(),dates.max(),freq='D'); pos=cal.get_indexer(dates)
valid=np.zeros(len(cal),bool);valid[pos]=True
A1=np.full((len(cal),b1.shape[1]),np.nan,np.float32);A2=np.full((len(cal),b2.shape[1]),np.nan,np.float32)
A1[pos]=b1; A2[pos]=b2
# causal forward fill (ONLY history, never target). All targets later read A1/A2 raw by exact date.
def causal_fill(raw,limit): return pd.DataFrame(raw).ffill(limit=limit).to_numpy(dtype=np.float32)
F1=causal_fill(A1,cfg.MAX_INPUT_GAP);F2=causal_fill(A2,cfg.MAX_INPUT_GAP)
annual=pd.DataFrame({'日期':cal,'有效':valid});annual['年份']=annual['日期'].dt.year
cover=annual.groupby('年份',as_index=False).agg(理论天数=('日期','size'),有效天数=('有效','sum'))
cover['有效比例']=cover['有效天数']/cover['理论天数'];cover.to_csv(DATA/'01_逐年覆盖率.csv',index=False,encoding='utf-8-sig')
if (cover['有效比例']>=cfg.MIN_ANNUAL_COVERAGE).sum()<cfg.MIN_DATA_YEARS:
    raise RuntimeError('具备≥85%有效覆盖的年份不足25年，请优先排查未完成人工修复或缺失年；详情见01_逐年覆盖率.csv')
hand=ROOT/cfg.HANDOFF_REL
if hand.exists():
    ho=json.loads(hand.read_text(encoding='utf-8'))
    print('HANDOFF final_daily_paths=',ho.get('final_daily_paths'))
    if int(ho.get('final_daily_paths',-1))!=len(dates): raise RuntimeError('HANDOFF与冻结npz样本数不一致')
else: warnings.warn('未发现HANDOFF.json；冻结数据可供资格检查，但交接记录尚未核实')
idxp=ROOT/cfg.QC_INDEX_REL
if idxp.exists():
    idx=pd.read_csv(idxp)
    if 'status' in idx.columns:
        unresolved=idx[~idx['status'].isin(['final','missing_raw','unusable'])]
        if len(unresolved): raise RuntimeError(f'最终逐日索引仍有 {len(unresolved)} 条未处理状态，停止训练')
queue=ROOT/cfg.MANUAL_QUEUE_REL; mdir=ROOT/cfg.MANUAL_METADATA_REL
if queue.exists():
    qq=pd.read_csv(queue)
    qdates=pd.to_datetime(qq['date']).dt.strftime('%Y%m%d') if len(qq) else []
    missing_meta=[q for q in qdates if not (mdir/f'{q}.json').exists()]
    if missing_meta: raise RuntimeError(f'{len(missing_meta)} 个manual复核队列日期尚无处理metadata: {missing_meta[:10]}')
    bad_meta=[]
    allowed={'manual_corrected','accepted_auto_after_manual_qc','unusable'}
    for q in qdates:
        try:
            state=json.loads((mdir/f'{q}.json').read_text(encoding='utf-8')).get('status')
            if state not in allowed:bad_meta.append((q,state))
        except Exception as e:bad_meta.append((q,repr(e)))
    if bad_meta:raise RuntimeError(f'{len(bad_meta)} 条人工QC仍不是已确认状态: {bad_meta[:10]}')
# CMEMS来源切换边界连续性检查：仅警告，不把真实大蛇行直接当成数据损坏
edge=pd.Timestamp('2025-01-01')
prev=np.flatnonzero(dates<edge)
post=np.flatnonzero(dates>=edge)
if len(prev) and len(post):
    i,j=int(prev[-1]),int(post[0])
    if (dates[j]-dates[i]).days<=5:
        d1=float(np.sqrt(np.mean((b1[j]-b1[i])**2)))
        d2=float(np.sqrt(np.mean((b2[j]-b2[i])**2)))
        pd.DataFrame([{'产品交界前':dates[i],'交界后':dates[j],
                       '分支1逐日经度RMSE_度':d1,'分支2逐日纬度RMSE_度':d2}]).to_csv(
            DATA/'01_2024到2025产品交界连续性.csv',index=False,encoding='utf-8-sig')
        if d1>1.0 or d2>1.5:
            warnings.warn('2024-2025 CMEMS来源交界发生较大路径跳变，应先检查跨产品物理一致性与流轴QC')
print('DATA PASS:',len(dates),'days;',dates.min().date(),'to',dates.max().date(),'; B1',b1.shape[1],'; B2',b2.shape[1])
display(cover)

In [ ]:
# ===== 3. 创建样本索引，不展开 [样本,180,特征] 巨型数组 =====
# target index = issue+lead; test targets require true valid day. input missing <=2 continuous days.
L=max(cfg.LEADS); H=cfg.HISTORY
missing=(~valid).astype(np.int32)
issue_indices=[];audit=[]
for i in range(H-1,len(cal)-L):
    if not valid[i]: continue
    win=missing[i-H+1:i+1]
    if win.mean()>cfg.MAX_MISSING_FRAC: continue
    if len(win) and np.max(np.diff(np.flatnonzero(np.r_[1,win==0,1]))-1)>cfg.MAX_INPUT_GAP: continue
    if any(not valid[i+d] for d in cfg.LEADS): continue
    if not (np.isfinite(F1[i-H+1:i+1]).all() and np.isfinite(F2[i-H+1:i+1]).all()): continue
    issue_indices.append(i)
ISS=np.asarray(issue_indices,np.int32)
if len(ISS)==0: raise RuntimeError('没有有效180→30/60/90 anchors，需检查缺日与冻结数据')
anchor=pd.DataFrame({'起报日期':cal[ISS]})
for l in cfg.LEADS: anchor[f'{l}天目标日期']=cal[ISS+l]
anchor.to_csv(DATA/'02_全部有效预测起报日期.csv',index=False,encoding='utf-8-sig')
print('anchor count=',len(ISS),' first=',cal[ISS[0]],'last=',cal[ISS[-1]])

In [ ]:
# ===== 4. 真正的 expanding-window 多年后报：test year = 2005/2009/2013/2017/2021/2024 =====
FOLDS=[];fr=[]; splitrows=[]
for fold_id,year in enumerate(cfg.TEST_YEARS,1):
    valstart=pd.Timestamp(year-1,1,1); teststart=pd.Timestamp(year,1,1)
    testend=pd.Timestamp(year,12,31)
    train_cut=valstart-pd.Timedelta(days=cfg.PURGE_DAYS)
    val_cut=teststart-pd.Timedelta(days=cfg.PURGE_DAYS)
    issues=cal[ISS]; latest=cal[ISS+L]
    tr=ISS[(latest<train_cut)]
    va=ISS[(issues>=valstart)&(issues<teststart)&(latest<val_cut)]
    te=ISS[(issues>=teststart)&(issues<=testend)]
    if len(tr)<cfg.MIN_TRAIN or len(va)<cfg.MIN_VAL or len(te)<cfg.MIN_TEST:
        raise RuntimeError(f'Fold {year} 样本不足: tr={len(tr)} val={len(va)} test={len(te)}；请查该年的日路径覆盖和连续缺日')
    if not(cal[tr+L].max()<cal[va].min()-pd.Timedelta(days=cfg.PURGE_DAYS) and cal[va+L].max()<cal[te].min()-pd.Timedelta(days=cfg.PURGE_DAYS)):
        raise RuntimeError(f'Fold {year} 发现严格时序泄漏')
    if (len(set(cal[tr+L])&set(cal[va+L])) or len(set(cal[tr+L])&set(cal[te+L])) or len(set(cal[va+L])&set(cal[te+L]))):
        raise RuntimeError(f'Fold {year} label date overlap')
    ff={'fold':fold_id,'year':year,'train':tr,'val':va,'test':te,'train_cut':train_cut,'val_cut':val_cut}
    FOLDS.append(ff)
    fr.append({'折次':fold_id,'测试年份':year,'训练样本':len(tr),'验证样本':len(va),'测试样本':len(te),
               '训练最晚标签日期':cal[tr+L].max(),'验证最早起报日期':cal[va].min(),
               '验证最晚标签日期':cal[va+L].max(),'测试最早起报日期':cal[te].min(),'测试最晚标签日期':cal[te+L].max(),'因果审计':'PASS'})
    for label,ix in [('训练',tr),('验证',va),('测试',te)]:
        splitrows.extend({'折次':fold_id,'测试年份':year,'集合':label,'起报日期':str(cal[i].date()),'最晚目标日期':str(cal[i+L].date())} for i in ix)
foldtab=pd.DataFrame(fr); foldtab.to_csv(DATA/'03_多年滚动后报折次及防泄漏审计.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(splitrows).to_csv(DATA/'03_逐样本时序划分.csv',index=False,encoding='utf-8-sig')
print('[STRICT MULTI-YEAR CAUSAL HINDCAST] ALL PASS'); display(foldtab)

## 阶段 0：加载原表示与训练主干

下方代码复用 R5.6 的 train-only EOF、卷积编码器、LSTM、训练优化器和路径 km 评价，保证实验更改仅涉及随机种子、评价年份与统计方式。

In [ ]:
# ===== 6. Original-only：不生成DP，不修改冻结数据 =====
A2_ORIGINAL=A2.copy()
F2_ORIGINAL=F2.copy()
print('ORIGINAL FROZEN REPRESENTATION ONLY; dataset SHA256:',sign)

In [ ]:
# ===== 5. 每折 train-only 标准化 + 自适应 99% EOF =====
# 使用该fold训练期的独立日记录，而不是重叠180d窗口重复叠加做SVD；内存恒定在 O(feature²)
class DailyEOF:
    def fit(self,obs,target=0.99):
        obs=np.asarray(obs,np.float64)
        self.mu=np.mean(obs,axis=0); self.sd=np.std(obs,axis=0); self.sd[self.sd<1e-6]=1.
        z=(obs-self.mu)/self.sd
        cov=(z.T@z)/max(1,len(z)-1)
        eig,vec=np.linalg.eigh(cov)
        idx=np.argsort(eig)[::-1];eig=np.maximum(eig[idx],0);vec=vec[:,idx]
        rat=np.cumsum(eig)/(eig.sum()+1e-12); n=int(np.searchsorted(rat,target)+1)
        self.V=np.asarray(vec[:,:n],np.float32);self.explained=float(rat[n-1]);self.n=n
        return self
    def pc(self,x): return (((np.asarray(x)-self.mu)/self.sd)@self.V).astype(np.float32)
    def invert(self,pc): return ((np.asarray(pc)@self.V.T)*self.sd+self.mu).astype(np.float32)
    def z(self,x): return ((np.asarray(x)-self.mu)/self.sd).astype(np.float32)

def make_fold_data(f):
    train_days=(cal< f['train_cut']) & valid
    pp1=DailyEOF().fit(A1[train_days],cfg.EOF_VARIANCE_TARGET)
    pp2=DailyEOF().fit(A2[train_days],cfg.EOF_VARIANCE_TARGET)
    for pp in (pp1,pp2):
        if pp.explained<cfg.EOF_VARIANCE_TARGET-1e-6: raise RuntimeError('EOF 99%训练方差Gate未通过')
    # approx memory: ~17k days * 2*(31+277) floats; never expands 180 day windows
    z1=pp1.z(F1);z2=pp2.z(F2)
    pcs1=pp1.pc(F1);pcs2=pp2.pc(F2)
    ppath=CKPT/ACTIVE_REP/f'第{f["fold"]}折_{f["year"]}年'; ppath.mkdir(exist_ok=True,parents=True)
    np.savez_compressed(ppath/'训练期预处理参数.npz',
        b1_mu=pp1.mu,b1_sd=pp1.sd,b1_V=pp1.V,b2_mu=pp2.mu,b2_sd=pp2.sd,b2_V=pp2.V,
        branch1_lat=lat1,branch2_lon=lon2,train_label_cutoff=str(f['train_cut'].date()))
    return {'pp1':pp1,'pp2':pp2,'z1':z1,'z2':z2,'p1':pcs1,'p2':pcs2,'path':ppath}

# 只预审一折，防止全局重复保留大数组
_testFD=make_fold_data(FOLDS[0]);print('fold1 EOF modes:',_testFD['pp1'].n,_testFD['pp2'].n,'explained:',_testFD['pp1'].explained,_testFD['pp2'].explained)
del _testFD;gc.collect()
print('[TRAIN-ONLY EOF 99%] PASS')

In [ ]:
# ===== 6. 动态加载数据集，训练时每个batch只展开当前180天窗口 =====
class CausalWindowDataset(Dataset):
    def __init__(self,issues,scaled,pcs):
        self.issues=np.asarray(issues,np.int32);self.scaled=scaled;self.pcs=pcs
    def __len__(self): return len(self.issues)
    def __getitem__(self,j):
        i=int(self.issues[j]);z=self.scaled[i-H+1:i+1]
        dz=np.empty_like(z);dz[0]=0;dz[1:]=z[1:]-z[:-1]
        x=np.concatenate([z,dz],axis=-1)
        y=np.stack([self.pcs[i+d] for d in cfg.LEADS])
        return torch.from_numpy(x.copy()),torch.from_numpy(y.copy())

class StepEncoder(nn.Module):
    # 与旧原稿的per-step TCN/Conv编码保持一致：输入路径特征卷积，然后LSTM承担时间建模
    def __init__(self,feature,branch,kind='tcn'):
        super().__init__();self.kind=kind;self.drop=nn.Dropout(cfg.DROPOUT)
        if kind=='tcn':
            channels=16 if branch==1 else 32;kernel=3 if branch==1 else 5
            self.conv=nn.Conv1d(1,channels,kernel,dilation=2)
            self.bn=nn.BatchNorm1d(channels)
            conv_len=feature+2-2*(kernel-1)
            self.fc=nn.Linear(channels*conv_len,32 if branch==1 else 64)
            self.out_dim=self.fc.out_features
        elif kind=='conv':
            c1=16 if branch==1 else 32;c2=32 if branch==1 else 64
            self.c1=nn.Conv1d(1,c1,3,padding=1);self.c2=nn.Conv1d(c1,c2,3,padding=1)
            self.out_dim=c2*feature
        else: raise ValueError(kind)
    def forward(self,x):
        B,T,F=x.shape; z=x.reshape(B*T,1,F)
        if self.kind=='tcn':
            z=torch.nn.functional.pad(z,(2,0)) # 与R3原架构一致的左侧填充
            z=self.drop(torch.relu(self.bn(self.conv(z))))
            z=self.drop(torch.relu(self.fc(z.flatten(1))))
        else:
            z=self.drop(torch.relu(self.c1(z)));z=self.drop(torch.relu(self.c2(z)));z=z.flatten(1)
        return z.reshape(B,T,-1)

class Hybrid(nn.Module):
    def __init__(self,feature,pc_dim,branch,kind='tcn',recur='lstm'):
        super().__init__();self.enc=StepEncoder(feature,branch,kind)
        h=32 if branch==1 else 64
        R=nn.GRU if recur=='gru' else nn.LSTM
        self.rnn=R(self.enc.out_dim,h,batch_first=True)
        self.fc=nn.Linear(h,3*pc_dim);self.pc_dim=pc_dim
    def forward(self,x):
        h,_=self.rnn(self.enc(x));return self.fc(h[:,-1]).reshape(len(x),3,self.pc_dim)

class MatchedLSTM(nn.Module):
    def __init__(self,feature,pc_dim,hidden):
        super().__init__();self.rnn=nn.LSTM(feature,hidden,batch_first=True)
        self.fc=nn.Linear(hidden,3*pc_dim);self.pc_dim=pc_dim
    def forward(self,x):
        h,_=self.rnn(x);return self.fc(h[:,-1]).reshape(len(x),3,self.pc_dim)

def nparams(m): return sum(p.numel() for p in m.parameters())
def match_lstm(feature,pc_dim,branch):
    target=nparams(Hybrid(feature,pc_dim,branch))
    # Pytorch single-layer LSTM: 4h(feature+h+2) + 3pc*h + 3pc
    opts=range(8,1025)
    h=min(opts,key=lambda hh:abs(4*hh*(feature+hh+2)+(3*pc_dim)*(hh+1)-target))
    actual=4*h*(feature+h+2)+(3*pc_dim)*(h+1)
    if abs(actual-target)/target>0.08: raise RuntimeError('Matched-LSTM参数量相对差>8%，不满足公平基线')
    return h,target,actual

def make_net(name,branch,fd):
    feature=fd[f'z{branch}'].shape[1]*2;pdim=fd[f'pp{branch}'].n
    if name=='Matched-LSTM':
        h,_,_=match_lstm(feature,pdim,branch);return MatchedLSTM(feature,pdim,h)
    if name=='EOF-TCN-LSTM': return Hybrid(feature,pdim,branch,'tcn','lstm')
    if name=='EOF-Conv-LSTM': return Hybrid(feature,pdim,branch,'conv','lstm')
    if name=='EOF-Conv-GRU': return Hybrid(feature,pdim,branch,'conv','gru')
    if name=='EOF-TCN-GRU': return Hybrid(feature,pdim,branch,'tcn','gru')
    raise ValueError(name)




# ===== R5.3：路径坐标/一阶差分分为两个空间输入通道 =====
class ChannelSeparatedEncoder(nn.Module):
    """单时刻空间卷积；输入[B,T,2*M] => Conv1d [B*T,2,M]."""
    def __init__(self,feature,branch):
        super().__init__()
        assert feature%2==0, '需要[x, delta]等长拼接输入'
        self.raw_dim=feature//2
        base_c=16 if branch==1 else 32
        k=3 if branch==1 else 5
        channels=base_c*2
        # 正确的对称空间填充：dilation=2 -> padding=k-1，输出仍为M
        self.conv=nn.Conv1d(2,channels,kernel_size=k,dilation=2,padding=k-1)
        self.bn=nn.BatchNorm1d(channels)
        self.fc=nn.Linear(channels*self.raw_dim,32 if branch==1 else 64)
        self.out_dim=self.fc.out_features
        self.drop=nn.Dropout(cfg.DROPOUT)
    def channelize(self,x):
        B,T,F=x.shape
        if F!=self.raw_dim*2: raise RuntimeError('R53特征维不匹配')
        return torch.stack((x[:,:,:self.raw_dim],x[:,:,self.raw_dim:]),dim=2).reshape(B*T,2,self.raw_dim)
    def forward(self,x):
        B,T,F=x.shape
        z=self.channelize(x)
        z=self.drop(torch.relu(self.bn(self.conv(z))))
        if z.shape[2]!=self.raw_dim:raise RuntimeError('R53 spatial-conv输出宽度发生变化')
        z=self.drop(torch.relu(self.fc(z.flatten(1))))
        return z.reshape(B,T,-1)

class ChannelHybrid(nn.Module):
    def __init__(self,feature,pc_dim,branch,projection=None,use_pc_skip=False):
        super().__init__()
        self.enc=ChannelSeparatedEncoder(feature,branch)
        self.use_pc_skip=bool(use_pc_skip)
        self.raw_dim=feature//2
        self.pc_dim=pc_dim
        if self.use_pc_skip:
            self.register_buffer('V',torch.as_tensor(projection,dtype=torch.float32))
        hidden=32 if branch==1 else 64
        in_dim=self.enc.out_dim+(2*pc_dim if self.use_pc_skip else 0)
        self.rnn=nn.LSTM(in_dim,hidden,batch_first=True)
        self.fc=nn.Linear(hidden,3*pc_dim)
    def eof_skip(self,x):
        if not self.use_pc_skip: raise RuntimeError('no PC skip in this model')
        return torch.cat((x[:,:,:self.raw_dim]@self.V,x[:,:,self.raw_dim:]@self.V),dim=-1)
    def forward(self,x):
        z=self.enc(x)
        if self.use_pc_skip: z=torch.cat((z,self.eof_skip(x)),dim=-1)
        h,_=self.rnn(z)
        return self.fc(h[:,-1]).reshape(len(x),3,self.pc_dim)

R53_CANDIDATES=('R53-ChannelConv','R53-ChannelConv-PCConcat')
R53_BASELINES=('Matched-LSTM','EOF-TCN-LSTM','R53-CapacityMatchedLSTM')
R53_ALL=R53_BASELINES+R53_CANDIDATES

def make_r53_model(name,branch,fd):
    feature=fd[f'z{branch}'].shape[1]*2
    pdim=fd[f'pp{branch}'].n
    if name in ('Matched-LSTM','EOF-TCN-LSTM'):
        return make_net(name,branch,fd)
    if name=='R53-CapacityMatchedLSTM':
        ref=ChannelHybrid(feature,pdim,branch,fd[f'pp{branch}'].V,use_pc_skip=True)
        h,_,_=matched_hidden_for_target(feature,pdim,nparams(ref))
        return MatchedLSTM(feature,pdim,h)
    if name=='R53-ChannelConv':
        return ChannelHybrid(feature,pdim,branch)
    if name=='R53-ChannelConv-PCConcat':
        return ChannelHybrid(feature,pdim,branch,fd[f'pp{branch}'].V,use_pc_skip=True)
    raise ValueError(name)

def matched_hidden_for_target(feature,pdim,target):
    # 单层 PyTorch LSTM 参数解析式，替代遍历建立 1017 个网络实例
    def count(h):return 4*h*(feature+h+2)+3*pdim*(h+1)
    best=min(range(8,1025),key=lambda h:abs(count(h)-target))
    actual=count(best)
    ratio=abs(actual-target)/max(target,1)
    if ratio>0.08:raise RuntimeError(f'参数匹配误差过大：{ratio:.2%}')
    return best,actual,ratio

def model_loss(pred,target,fd,branch,name):
    # 与R5.2完全相同的训练目标：只用PC-MSE做结构归因。
    return torch.nn.functional.mse_loss(pred,target)

In [ ]:
# ===== 8. 物理公里误差：局地投影+最近点相邻线段的双向正交路径距离 =====
# 对于仅122-145E的分支2，重建得到共同定义域；不是对每一点简单经纬度平方误差。
# 下方是双向point-to-polyline近似cross-track，使用KDTree近邻三顶点及其邻接segments。
def merge_path(branch1,branch2):
    a=np.asarray(branch1,float);b=np.asarray(branch2,float)
    candidates=np.flatnonzero(lon2>a[-1]);
    k=int(candidates[np.argmin((lon2[candidates]-a[-1])**2+(b[candidates]-lat1[-1])**2)]) if len(candidates) else len(lon2)
    xs=np.r_[a,lon2[k:]];ys=np.r_[lat1,b[k:]]
    good=np.isfinite(xs)&np.isfinite(ys)
    return xs[good],ys[good]

def one_direction(A,B):
    # A,B: 2d coordinates (km); approximate nearest polyline segment for each A point
    if len(A)<2 or len(B)<2: return np.array([np.nan])
    tree=cKDTree(B);_,knn=tree.query(A,k=min(3,len(B)))
    knn=np.asarray(knn);knn=knn[:,None] if knn.ndim==1 else knn
    ix=np.unique(np.clip(np.concatenate([knn.ravel()-1,knn.ravel()]),0,len(B)-2))
    # segment shortlist globally may grow, but polynomial per path length (~300) and is vectorized
    v=B[ix+1]-B[ix];w=A[:,None,:]-B[ix][None,:,:]
    den=np.maximum(np.sum(v*v,axis=1),1e-12)
    t=np.clip(np.sum(w*v[None,:,:],axis=2)/den[None,:],0,1)
    q=B[ix][None,:,:]+t[:,:,None]*v[None,:,:]
    d=np.sqrt(np.sum((A[:,None,:]-q)**2,axis=2))
    return d.min(axis=1)

def errors_for_pair(pred1,pred2,truth1,truth2):
    x,y=merge_path(pred1,pred2);xt,yt=merge_path(truth1,truth2)
    lat0=np.mean(np.r_[y,yt]);lon0=np.mean(np.r_[x,xt]);c=np.cos(np.deg2rad(lat0))
    A=np.column_stack([(x-lon0)*111.32*c,(y-lat0)*111.32]);B=np.column_stack([(xt-lon0)*111.32*c,(yt-lat0)*111.32])
    d=np.r_[one_direction(A,B),one_direction(B,A)]
    return float(np.mean(d**2)),float(np.mean(d)),float(np.percentile(d,90))

def record_errors(fold,model,seed,issues,p1,p2):
    out=[]
    for n,i in enumerate(issues):
        for k,lead in enumerate(cfg.LEADS):
            y1=A1[i+lead];y2=A2[i+lead]
            mse,mean,p90=errors_for_pair(p1[n,k],p2[n,k],y1,y2)
            out.append({'折次':fold['fold'],'测试年份':fold['year'],'模型':model,'种子':seed,'起报日期':str(cal[i].date()),'目标日期':str(cal[i+lead].date()),'提前期_天':int(lead),
                        '横向路径误差MSE_km2':mse,'横向路径平均误差_km':mean,'横向路径P90_km':p90,
                        '分支1经度RMSE_度':float(np.sqrt(np.mean((p1[n,k]-y1)**2))),
                        '分支2纬度RMSE_度':float(np.sqrt(np.mean((p2[n,k]-y2)**2)))})
    return pd.DataFrame(out)

In [ ]:
# ===== 9. R5 训练封装、物理评价、可恢复 checkpoints =====
from contextlib import nullcontext

def r5_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)

def run_predict(net, issues, fd, branch):
    ds=CausalWindowDataset(issues,fd[f'z{branch}'],fd[f'p{branch}'])
    ld=DataLoader(ds,batch_size=cfg.BATCH_SIZE,shuffle=False,num_workers=0)
    arr=[];net.eval()
    with torch.no_grad():
        for xx,_ in ld: arr.append(net(xx.to(DEV)).float().cpu().numpy())
    return np.concatenate(arr,axis=0)

def fit_r5_branch(split, fd, name, seed, branch, root_dir, max_epochs=None, patience=None):
    max_epochs=cfg.PILOT_EPOCHS if max_epochs is None else max_epochs
    patience=cfg.PILOT_PATIENCE if patience is None else patience
    folder=root_dir/ACTIVE_REP/f'QA_{split["year"]-1}'/name/f'seed_{seed}'
    folder.mkdir(parents=True,exist_ok=True)
    weight=folder/f'branch{branch}_best.pt'; historyfile=folder/f'branch{branch}_history.csv'
    latest=folder/f'branch{branch}_latest.pt'
    # best checkpoint 附带实验合同签名；仅同一fold/样本划分/网络版本可复用
    contract={'name':name,'seed':seed,'branch':branch,'tr_n':int(len(split['train'])),
              'val_n':int(len(split['val'])),'tr_last':int(split['train'][-1]),
              'val_first':int(split['val'][0]),'val_last':int(split['val'][-1]),'train_first':int(split['train'][0]),'max_epochs':max_epochs,
              'feature_dims':[fd['z1'].shape[1],fd['z2'].shape[1]],'version':'R5.7-original-stability-v1','rep':ACTIVE_REP,'dataset_signature':sign}
    contract_id=hashlib.sha256(json.dumps(contract,sort_keys=True).encode()).hexdigest()
    r5_seed(seed)
    net=make_r53_model(name,branch,fd).to(DEV)
    if weight.exists() and historyfile.exists() and not cfg.FORCE_TRAIN:
        ck=torch.load(weight,map_location='cpu',weights_only=True)
        if ck.get('contract_id')==contract_id:
            net.load_state_dict(ck['state']);return net,pd.read_csv(historyfile)
    ds_tr=CausalWindowDataset(split['train'],fd[f'z{branch}'],fd[f'p{branch}'])
    ds_va=CausalWindowDataset(split['val'],fd[f'z{branch}'],fd[f'p{branch}'])
    ld_tr=DataLoader(ds_tr,batch_size=cfg.BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=True)
    ld_va=DataLoader(ds_va,batch_size=cfg.BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
    opt=torch.optim.AdamW(net.parameters(),lr=cfg.LR,weight_decay=cfg.WEIGHT_DECAY)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,factor=0.5,patience=4)
    amp=bool(cfg.USE_AMP)
    scaler=torch.amp.GradScaler('cuda',enabled=amp)
    start_epoch=1;best=float('inf');bestep=0;bad=0;hist=[]
    if latest.exists() and not cfg.FORCE_TRAIN:
        try:
            ck=torch.load(latest,map_location='cpu',weights_only=True)
            if ck.get('contract_id')==contract_id:
                net.load_state_dict(ck['state']);opt.load_state_dict(ck['opt']);scheduler.load_state_dict(ck['scheduler']);scaler.load_state_dict(ck['scaler'])
                start_epoch=int(ck['epoch'])+1;best=float(ck['best']);bestep=int(ck['bestep']);bad=int(ck['bad']);hist=ck['hist']
                print('[RESUME]',folder,'epoch',start_epoch)
        except Exception as e:print('[resume invalid]',e)
    for ep in range(start_epoch,max_epochs+1):
        net.train();ts=tn=0
        for x,y in ld_tr:
            x=x.to(DEV,non_blocking=True);y=y.to(DEV,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda',enabled=amp):
                pred=net(x)
                loss=model_loss(pred.float(),y.float(),fd,branch,name)
            if not torch.isfinite(loss):raise RuntimeError(f'{name} B{branch}: non-finite loss')
            scaler.scale(loss).backward();scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(),1.0)
            scaler.step(opt);scaler.update();ts+=float(loss.detach())*len(x);tn+=len(x)
        net.eval();vs=vn=0
        with torch.no_grad():
            for x,y in ld_va:
                x=x.to(DEV);y=y.to(DEV)
                with torch.autocast('cuda',enabled=amp):
                    p=net(x);loss=model_loss(p.float(),y.float(),fd,branch,name)
                vs+=float(loss.detach())*len(x);vn+=len(x)
        val=vs/max(1,vn);scheduler.step(val)
        hist.append({'epoch':ep,'train_loss':ts/max(1,tn),'val_loss':val,'lr':opt.param_groups[0]['lr']})
        if val<best-1e-7:
            best=val;bestep=ep;bad=0
            torch.save({'state':{k:v.detach().cpu() for k,v in net.state_dict().items()},'contract_id':contract_id,'epoch':ep,'best_val':best},weight)
        else:bad+=1
        if ep%5==0 or bad>=patience or ep==max_epochs:
            torch.save({'state':{k:v.detach().cpu() for k,v in net.state_dict().items()},'opt':opt.state_dict(),
                        'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),'epoch':ep,'best':best,
                        'bestep':bestep,'bad':bad,'hist':hist,'contract_id':contract_id},latest)
        if ep==1 or ep%10==0 or bad>=patience:print(f'{name} B{branch} seed{seed} ep{ep} train={hist[-1]["train_loss"]:.5f} val={val:.5f}')
        if bad>=patience:break
    pd.DataFrame(hist).to_csv(historyfile,index=False,encoding='utf-8-sig')
    if latest.exists():latest.unlink()
    ck=torch.load(weight,map_location='cpu',weights_only=True)
    net.load_state_dict(ck['state']);net.eval()
    return net,pd.DataFrame(hist)

def evaluate_issues(f,name,seed,fd,issues,bmodel1,bmodel2):
    return record_errors(f,name,seed,issues,bmodel1,bmodel2)

def get_predictions(name,split,fd,seed,folder,max_epochs=None,patience=None):
    branches=[]
    for branch in (1,2):
        net,hist=fit_r5_branch(split,fd,name,seed,branch,folder,max_epochs,patience)
        pr=run_predict(net,split.get('qa',split.get('test')),fd,branch)
        branches.append(fd[f'pp{branch}'].invert(pr))
        del net,pr;torch.cuda.empty_cache()
    return branches

In [ ]:
# ===== 7. 预先固定 QA 拆分：标签与早停严格先于 QA 日期 =====
PILOT_QA_YEARS = (2008,2016)
CONFIRM_QA_YEARS = (2010,2018)
R57_MODELS = ('Matched-LSTM','R53-CapacityMatchedLSTM','EOF-TCN-LSTM','R53-ChannelConv-PCConcat')
R57_SEEDS=tuple(cfg.PILOT_SEEDS)

# QA 年份不再依赖 R56 TEST_YEARS 白名单；其余合同完全一致。
def r57_pilot_split(qa_year):
    qa_start=pd.Timestamp(qa_year,1,1)
    next_start=pd.Timestamp(qa_year+1,1,1)
    inner_start=pd.Timestamp(qa_year-1,1,1)
    purge=pd.Timedelta(days=cfg.PURGE_DAYS)
    train=ISS[cal[ISS+L] < inner_start-purge]
    val=ISS[(cal[ISS]>=inner_start) & (cal[ISS+L]<qa_start-purge)]
    qa=ISS[(cal[ISS]>=qa_start) & (cal[ISS+L]<next_start-purge)]
    if min(len(train),len(val),len(qa))<min(cfg.MIN_TRAIN,cfg.MIN_VAL):
        raise RuntimeError(f'QA {qa_year} 样本量不足 train={len(train)} val={len(val)} qa={len(qa)}')
    if not (len(train)>=cfg.MIN_TRAIN and len(val)>=cfg.MIN_VAL and len(qa)>=cfg.MIN_VAL):
        raise RuntimeError(f'QA {qa_year} 数据资格未满足')
    assert cal[train+L].max()<cal[val].min()-purge
    assert cal[val+L].max()<cal[qa].min()-purge
    return {'fold':qa_year,'year':qa_year+1,'train':train,'val':val,'qa':qa,'train_cut':inner_start-purge}

PILOTS=[r57_pilot_split(y) for y in PILOT_QA_YEARS]
CONFIRMS=[r57_pilot_split(y) for y in CONFIRM_QA_YEARS]
SPLIT_ROWS=[]
for phase,fs in [('资格探索',PILOTS),('后续年份确认',CONFIRMS)]:
    for f in fs:
        SPLIT_ROWS.append({'阶段':phase,'QA年份':f['year']-1,'训练数':len(f['train']),'早停数':len(f['val']),'QA数':len(f['qa']),
          '训练最晚标签日期':str(cal[f['train']+L].max().date()),'早停最晚标签日期':str(cal[f['val']+L].max().date()),
          'QA最早起报日期':str(cal[f['qa']].min().date())})
split_df=pd.DataFrame(SPLIT_ROWS)
split_df.to_csv(DATA/'R57_01_严格因果时序拆分.csv',index=False,encoding='utf-8-sig')
display(split_df)
print('[R57 STRICT CAUSAL SPLIT] PASS')

In [ ]:
# ===== 8. 训练前模型形状、参数匹配与数据资格验证 =====
fd0=make_fold_data(PILOTS[0]); param_rows=[]
for branch in (1,2):
    fdim=fd0[f'z{branch}'].shape[1]*2
    for name in R57_MODELS:
        model=make_r53_model(name,branch,fd0).to(DEV)
        x=torch.randn(2,H,fdim,device=DEV)
        y=model(x)
        assert y.shape==(2,3,fd0[f'pp{branch}'].n) and torch.isfinite(y).all()
        y.square().mean().backward()
        param_rows.append({'模型':name,'分支':branch,'参数量':nparams(model),'EOF维度':fd0[f'pp{branch}'].n})
        del model
pd.DataFrame(param_rows).to_csv(DATA/'R57_02_模型参数量.csv',index=False,encoding='utf-8-sig')
display(pd.DataFrame(param_rows))
assert np.isfinite(A2[valid]).all() and np.array_equal(A2,A2_ORIGINAL,equal_nan=True)
print('[R57 GPU/SHAPE/GRAD/ORIGINAL-GUARD] ALL PASS')
del fd0;torch.cuda.empty_cache();gc.collect()

In [ ]:
# ===== 9. 自动打包：任何资格结果、失败或中断均可归档 =====
import zipfile,traceback,hashlib
from IPython.display import FileLink,display,Markdown
ARCHIVE_INCLUDE={'.csv','.json','.txt','.md','.png','.pdf','.log'}
def archive_r57(stage,exception=None):
    state={'experiment':'R5.7 Original-only seed-stability','stage':stage,
           'timestamp_utc':pd.Timestamp.now(tz='UTC').isoformat(),
           'dataset_sha256':sign,'QA_exploratory_years':PILOT_QA_YEARS,
           'QA_confirmation_years':CONFIRM_QA_YEARS,'seeds':R57_SEEDS,
           'checkpoint_archived':False,'repeated_exposure_not_blind':True,
           'exception':exception}
    (REP/'R57_阶段状态.json').write_text(json.dumps(state,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
    files=[]
    for d in (DATA,REP,FZH,FEN):
        for f in d.rglob('*'):
            if f.is_file() and f.suffix.lower() in ARCHIVE_INCLUDE and f.stat().st_size<150*1024*1024:
                files.append(f)
    manifest=[]
    for f in sorted(set(files)):
        manifest.append({'相对路径':str(f.relative_to(OUT)),'字节数':f.stat().st_size,
                         'SHA256':hashlib.sha256(f.read_bytes()).hexdigest()})
    man=REP/'R57_归档文件清单.csv'
    pd.DataFrame(manifest).to_csv(man,index=False,encoding='utf-8-sig')
    if man not in files:files.append(man)
    tmp=Path(str(ARCHIVE_PATH)+'.tmp')
    with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as zz:
        for f in sorted(set(files)):
            zz.write(f,arcname=str(Path('R57_结果')/f.relative_to(OUT)))
        zz.writestr('R57_结果/README.txt', 'R5.7 原表示模型稳定性资格实验。完整模型权重、预测大缓存不在此分享包中；包含逐样本统计与分层分析。')
    with zipfile.ZipFile(tmp) as zz:
        assert zz.testzip() is None
    tmp.replace(ARCHIVE_PATH)
    digest=hashlib.sha256(ARCHIVE_PATH.read_bytes()).hexdigest()
    ARCHIVE_SHA.write_text(f'{digest}  {ARCHIVE_PATH.name}\n',encoding='utf-8')
    print('[R57 ZIP READY]',ARCHIVE_PATH,'size MB=',round(ARCHIVE_PATH.stat().st_size/1048576,2))
    print('[SHA256]',digest)
    display(FileLink(str(ARCHIVE_PATH)))
    display(Markdown(f'**Windows 文件**：`D:\\KuroshioDaily1993_2025\\{ARCHIVE_PATH.name}`。若 VS Code 无法打开上方链接，在 Windows 资源管理器直接访问此路径并上传 ZIP。'))
    return ARCHIVE_PATH

print('[R57 ARCHIVER READY]')

In [ ]:
# ===== 10. 机制 QA：所有预测均评价在同一冻结原表示上 =====
CACHE=OUT/'资格预测缓存';CACHE.mkdir(parents=True,exist_ok=True)
QA_PATH=DATA/'R57_03_逐样本共同冻结评价.csv'
# cache key 同时包含 split、模型、seed、协议及数据SHA；避免错误复用旧预测。
def r57_cache_signature(f,name,seed):
    raw={'year':f['year'],'train':len(f['train']),'train_last':int(f['train'][-1]),
         'val':len(f['val']),'qa':len(f['qa']),'name':name,'seed':seed,
         'contract':'R57-original-stability-v1','data_sha':sign,'max_epochs':cfg.PILOT_EPOCHS,
         'patience':cfg.PILOT_PATIENCE,'EOF_variance':cfg.EOF_VARIANCE_TARGET}
    return hashlib.sha256(json.dumps(raw,sort_keys=True).encode()).hexdigest()

def records_for_predictions(f,name,seed,p1,p2,phase):
    all_rows=[]
    for n,ii in enumerate(f['qa']):
        ii=int(ii)
        for k,lead in enumerate(cfg.LEADS):
            j=ii+lead
            mse,mean_dist,p90=errors_for_pair(p1[n,k],p2[n,k],A1[j],A2_ORIGINAL[j])
            all_rows.append({'阶段':phase,'QA年份':f['year']-1,'模型':name,'种子':seed,
               '起报日期':str(cal[ii].date()),'目标日期':str(cal[j].date()),
               '提前期_天':int(lead),'冻结目标MSE_km2':float(mse),
               '冻结目标平均距离_km':float(mean_dist),'冻结目标P90_km':float(p90),'物理范围异常目标日':bool(suspicious[np.searchsorted(dates.values,cal[j].to_datetime64())]) if valid[j] else False})
    return pd.DataFrame(all_rows)

def train_eval_phase(folds,phase):
    groups=[]
    if QA_PATH.exists():
        history=pd.read_csv(QA_PATH)
        groups.append(history[history['阶段']!=phase])
    outrows=[]
    for f in folds:
        fd=make_fold_data(f)
        try:
            for model in R57_MODELS:
                for seed in R57_SEEDS:
                    cf=CACHE/f'{phase}_{f["year"]}_{model}_seed{seed}_{r57_cache_signature(f,model,seed)[:12]}.npz'
                    if cf.exists() and not cfg.FORCE_TRAIN:
                        with np.load(cf,allow_pickle=False) as z:
                            p1,p2=np.asarray(z['b1']),np.asarray(z['b2'])
                            ids=np.asarray(z['issues'])
                        if not np.array_equal(ids,f['qa']):raise RuntimeError('cache QA date mismatch: '+str(cf))
                        print('[R57 RESUME PREDICTIONS]',cf.name)
                    else:
                        p1,p2=get_predictions(model,f,fd,seed,OUT/'模型权重',cfg.PILOT_EPOCHS,cfg.PILOT_PATIENCE)
                        np.savez_compressed(cf,b1=p1,b2=p2,issues=f['qa'])
                    outrows.append(records_for_predictions(f,model,seed,p1,p2,phase))
                    combined=pd.concat(groups+outrows,ignore_index=True)
                    combined.to_csv(QA_PATH,index=False,encoding='utf-8-sig')
                    print('[R57]',phase,'QA',f['year']-1,model,'seed',seed,'samples',len(f['qa']))
        finally:
            del fd;gc.collect();torch.cuda.empty_cache()
    archive_r57('completed_'+phase)
    return pd.concat(groups+outrows,ignore_index=True)

def summarize_phase(df,phase):
    d=df[df['阶段']==phase].copy()
    expected_years=PILOT_QA_YEARS if phase=='探索QA' else CONFIRM_QA_YEARS
    expected_keys={(yr,model,seed,lead) for yr in expected_years for model in R57_MODELS
                   for seed in R57_SEEDS for lead in cfg.LEADS}
    actual_keys=set(zip(d['QA年份'].astype(int),d['模型'],d['种子'].astype(int),d['提前期_天'].astype(int)))
    if expected_keys!=actual_keys:
        missing=expected_keys-actual_keys;unexpected=actual_keys-expected_keys
        raise RuntimeError(f'R57 {phase} 不完整，禁止判 Gate: 缺失组 {list(missing)[:8]}；额外组 {list(unexpected)[:8]}')
    expected_per_year={f['year']-1:len(f['qa']) for f in (PILOTS if phase=='探索QA' else CONFIRMS)}
    group_sizes=d.groupby(['QA年份','模型','种子','提前期_天']).size()
    for (yr,model,seed,lead),size in group_sizes.items():
        if int(size)!=expected_per_year[int(yr)]:
            raise RuntimeError(f'R57 {phase} 样本组数不符 {yr}/{model}/{seed}/{lead}: {size} != {expected_per_year[int(yr)]}')
    r=d.groupby(['阶段','QA年份','模型','种子','提前期_天'],as_index=False).agg(
       RMSE_km=('冻结目标MSE_km2',lambda x:float(np.sqrt(np.mean(x))))
    )
    r.to_csv(DATA/f'R57_04_{phase}_逐年份种子指标.csv',index=False,encoding='utf-8-sig')
    means=r.groupby(['模型','提前期_天'],as_index=False)['RMSE_km'].mean()
    means.to_csv(DATA/f'R57_04_{phase}_平均性能.csv',index=False,encoding='utf-8-sig')
    wide=r.pivot(index=['QA年份','种子','提前期_天'],columns='模型',values='RMSE_km')
    names=set(wide.columns)
    if not set(R57_MODELS).issubset(names):raise RuntimeError('资格结果不完整，不得判定 Gate')
    wide['强基线RMSE']=wide[['Matched-LSTM','R53-CapacityMatchedLSTM']].min(axis=1)
    wide['混合相对强LSTM改善_km']=wide['强基线RMSE']-wide['R53-ChannelConv-PCConcat']
    wide['混合相对原ETL改善_km']=wide['EOF-TCN-LSTM']-wide['R53-ChannelConv-PCConcat']
    pairs=wide.reset_index()
    pairs.to_csv(DATA/f'R57_05_{phase}_逐年份种子提前期成对改善.csv',index=False,encoding='utf-8-sig')
    gain=pairs['混合相对强LSTM改善_km']
    yg=pairs.groupby('QA年份')['混合相对强LSTM改善_km'].mean()
    lg=pairs.groupby('提前期_天')['混合相对强LSTM改善_km'].mean()
    sg=pairs.groupby(['QA年份','种子'])['混合相对强LSTM改善_km'].mean()
    status={'阶段':phase,'年份':sorted(map(int,yg.index)),'种子':list(R57_SEEDS),
       '混合相对强LSTM平均改善_km':float(gain.mean()),
       '混合相对原ETL平均改善_km':float(pairs['混合相对原ETL改善_km'].mean()),
       '逐年平均改善_km':{str(k):float(v) for k,v in yg.items()},
       '逐提前期平均改善_km':{str(k):float(v) for k,v in lg.items()},
       '每年种子组改善胜率':float((sg>0).mean()),
       '所有年份正改善':bool((yg>0).all()),
       '各提前期不退化':bool((lg>=0).all()),
       '种子组至少三分之二胜出':bool((sg>0).mean()>=2/3),
       '平均改善至少0.5km':bool(gain.mean()>=0.5),
       '对原ETL有改善':bool(pairs['混合相对原ETL改善_km'].mean()>0)}
    status['通过']=bool(status['所有年份正改善'] and status['各提前期不退化'] and
      status['种子组至少三分之二胜出'] and status['平均改善至少0.5km'] and status['对原ETL有改善'])
    (REP/f'R57_05_{phase}_资格判定.json').write_text(json.dumps(status,indent=2,ensure_ascii=False),encoding='utf-8')
    display(means);display(pairs.groupby(['QA年份','提前期_天'])['混合相对强LSTM改善_km'].mean().reset_index())
    print('[R57 GATE]',phase,json.dumps(status,ensure_ascii=False,indent=2))
    return status

print('[R57 EVAL ENGINE READY]')

In [ ]:
# ===== 11. 阶段一：三新种子 × 2008/2016 × 四模型 =====
try:
    R57_QA=train_eval_phase(PILOTS,'探索QA')
    GATE_QA=summarize_phase(R57_QA,'探索QA')
    archive_r57('exploratory_gate_completed')
except Exception:
    (REP/'R57_异常追踪.txt').write_text(traceback.format_exc(),encoding='utf-8')
    archive_r57('exploratory_exception',traceback.format_exc())
    raise

In [ ]:
# ===== 12. 阶段二：只有探索QA真正通过时，才执行2010/2018验证 =====
GATE_CONFIRM={'通过':False,'原因':'探索阶段未通过或配置禁用；未运行额外确认'}
if not GATE_QA['通过']:
    print('[R57 STOP] 探索QA不达标；不运行额外年份。保留全部失败证据。')
elif not cfg.RUN_CONFIRMATION:
    print('[R57 HOLD] 探索QA通过，但 RUN_CONFIRMATION=False；由研究者控制。')
else:
    try:
        R57_QA=train_eval_phase(CONFIRMS,'额外确认')
        GATE_CONFIRM=summarize_phase(R57_QA,'额外确认')
        archive_r57('confirmation_gate_completed')
    except Exception:
        (REP/'R57_异常追踪.txt').write_text(traceback.format_exc(),encoding='utf-8')
        archive_r57('confirmation_exception',traceback.format_exc())
        raise

In [ ]:
# ===== 13. 中英文图表 + 最终研究结论 + ZIP / 可点击链接 =====
full=pd.read_csv(QA_PATH) if QA_PATH.exists() else pd.DataFrame()
if len(full):
    for phase in full['阶段'].dropna().unique():
        tmp=full[full['阶段']==phase]
        stats=tmp.groupby(['模型','QA年份','种子','提前期_天'])['冻结目标MSE_km2'].mean().pow(0.5).reset_index(name='RMSE_km')
        avg=stats.groupby(['模型','提前期_天'])['RMSE_km'].mean().reset_index()
        palette={'Matched-LSTM':'#729cbb','R53-CapacityMatchedLSTM':'#93b7ca','EOF-TCN-LSTM':'#566f95','R53-ChannelConv-PCConcat':'#d08a58'}
        for zh,target in ((True,FZH),(False,FEN)):
            old=plt.rcParams.copy();plt.rcParams['font.family']=[CJK_NAME] if zh else ['DejaVu Sans']
            fig,ax=plt.subplots(figsize=(8.0,4.5))
            for model in R57_MODELS:
                sub=avg[avg['模型']==model].sort_values('提前期_天')
                ax.plot(sub['提前期_天'],sub['RMSE_km'],color=palette[model],marker='o',lw=1.7,label=model)
            ax.set_xlabel('提前期（天）' if zh else 'Forecast lead (days)')
            ax.set_ylabel('冻结流轴路径 RMSE (km)' if zh else 'Frozen-reference path RMSE (km)')
            ax.set_title('R5.7 原表示模型稳定性对比' if zh else 'R5.7 Original-representation model comparison')
            ax.grid(alpha=.2);ax.legend(fontsize=8);fig.tight_layout()
            suf='中文' if zh else 'English'
            fig.savefig(target/f'R57_{phase}_模型稳定性_{suf}.png',dpi=cfg.FIG_DPI,bbox_inches='tight')
            fig.savefig(target/f'R57_{phase}_模型稳定性_{suf}.pdf',bbox_inches='tight')
            plt.close(fig);plt.rcParams.update(old)
summary={'experiment':'R57-original-stability-v1','data_immutable':True,'exploratory_gate':GATE_QA,
    'confirmation_gate':GATE_CONFIRM,'formal_multi_year_training_qualified':bool(GATE_QA['通过'] and GATE_CONFIRM.get('通过',False)),
    'note':'曾多次查看原多年份结果；本轮只是新增种子与时间段重复验证，不能作为真正预注册的全盲测试。未通过时不得声称混合模型全面优于强LSTM。'}
(REP/'R57_最终交接.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
archive_r57('completed_evaluated')

## 结果解读规则

必须以 `R57_最终交接.json` 和各阶段 `R57_05_*_资格判定.json` 为准。**即使探索 QA 通过，也不代表已经具备正式发表的独立盲测泛化证据。** 本轮不改变 Original 冻结真值，不选择性剔除已标注物理范围异常的日期。实验包自动打包后直接上传本对话。